In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully! Ready for TCGA data analysis.")

In [ ]:
# Load your TCGA dataset (replace 'tcga_data.csv' with your actual file name if needed)
# For now, let's write the template structure:

# df = pd.read_csv('tcga_data.csv')
# print(df.head())

print("Data loading template ready. Drop your dataset into your working folder or provide your filename!")

In [ ]:
# Load your uploaded CSV file
df = pd.read_csv('TCGA_Model_Predictions.csv')

# Display the first 5 rows of your dataset
print("Dataset successfully loaded!")
display(df.head())

In [ ]:
# Check the first few column names to see how Orange structured the headers
print("First 10 columns:")
print(df.columns[:10])

# Check rows 0 and 1 to see if we need to clean up extra header rows
print("\nPreview of header rows:")
display(df.head(2))

In [ ]:
# Drop the first two rows (index 0 and 1) which contain Orange metadata tags
df_clean = df.drop([0, 1]).reset_index(drop=True)

# Display the first 5 rows of your cleaned dataframe
print(f"Cleaned dataset shape: {df_clean.shape}")
display(df_clean.head())

In [ ]:
# Let's inspect the target column name and unique values to be sure
print("Target column unique values:")
print(df_clean['OS_STATUS'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

# Clean and map the target variable to integers (0 and 1)
# Checking if it contains strings like '0:LIVING'
if df_clean['OS_STATUS'].dtype == 'object':
    df_clean['target'] = df_clean['OS_STATUS'].apply(lambda x: 1 if '1' in str(x) else 0)
else:
    df_clean['target'] = df_clean['OS_STATUS']

# Select features (dropping metadata columns, target columns, and previous model predictions to avoid data leakage)
drop_cols = ['OS_STATUS', 'target', 'studyId:sampleId', 'patientId', 'OTHER_PATIENT_ID', 
             'Random Forest', 'Logistic Regression', 'SVM']

# Keep only columns that actually exist in the dataframe
features_to_drop = [col for col in drop_cols if col in df_clean.columns]
X = df_clean.drop(columns=features_to_drop)

# Convert all feature columns to numeric (coercing non-numeric to NaN, then filling with 0)
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

y = df_clean['target']

print(f"Features shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")
print(f"Class breakdown in target:\n{y.value_counts()}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Initialize the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 3. Train the model on your TCGA data
rf_model.fit(X_train, y_train)

# 4. Make predictions on the test set
y_pred = rf_model.predict(X_test)

# 5. Evaluate model performance
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Get feature importances from the trained model
importances = rf_model.feature_importances_
feature_names = X.columns

# Create a dataframe for visualization
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).head(10)

# Plot the top 10 features
plt.figure(figsize=(10, 6))
plt.barh(feature_imp_df['Feature'][::-1], feature_imp_df['Importance'][::-1], color='teal')
plt.xlabel('Feature Importance Score')
plt.ylabel('Features')
plt.title('Top 10 Important Features in TCGA Survival Model')
plt.tight_layout()
plt.show()

In [ ]:
# Filter out any columns that belong to model predictions or IDs
cols_to_exclude = [col for col in X.columns if any(model_name in col for model_name in ['Random Forest', 'Logistic Regression', 'SVM', 'id', 'ID'])]
X_true = X.drop(columns=[col for col in cols_to_exclude if col in X.columns])

print(f"Cleaned feature matrix shape (excluding model predictions): {X_true.shape}")

# Re-split and re-train the model without data leakage
X_train, X_test, y_train, y_test = train_test_split(X_true, y, test_size=0.2, random_state=42, stratify=y)

rf_clean_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clean_model.fit(X_train, y_train)

# Plot true feature importances
importances = rf_clean_model.feature_importances_
feature_names = X_true.columns

feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(feature_imp_df['Feature'][::-1], feature_imp_df['Importance'][::-1], color='darkorange')
plt.xlabel('Feature Importance Score')
plt.ylabel('Features')
plt.title('Top 10 True Clinical & Multi-Omics Biomarkers for Survival')
plt.tight_layout()
plt.show()